# Extracción y construcción del dataset académico

Este notebook procesa fichas de rendimiento académico en formato PDF y genera un conjunto estructurado de registros académicos para análisis de trayectorias.

In [1]:
from pathlib import Path
import re
import pandas as pd

from pypdf import PdfReader

print("Entorno listo")

Entorno listo


In [2]:
BASE_DIR = Path("../")

DATA_RAW = BASE_DIR / "data" / "raw" / "pdf_estudiantes"

DATA_INTERIM = BASE_DIR / "data" / "interim"

print(DATA_RAW)

..\data\raw\pdf_estudiantes


In [3]:
def leer_pdf(ruta_pdf):
    
    reader = PdfReader(ruta_pdf)

    texto = ""

    for pagina in reader.pages:
        contenido = pagina.extract_text()

        if contenido:
            texto += contenido + "\n"

    return texto

In [4]:
archivos = list(DATA_RAW.glob("*.pdf"))

print(
    f"PDF encontrados: {len(archivos)}"
)

PDF encontrados: 247


In [5]:
def extraer_datos_generales(texto):

    datos = {}

    codigo = re.search(
        r"CÓDIGO / MATRÍCULA\s+(\d+)",
        texto
    )

    datos["id_estudiante"] = (
        codigo.group(1)
        if codigo else None
    )


    programa = re.search(
        r"PROGRAMA DE ESTUDIOS\s+(.*?)\s+APELLIDOS Y NOMBRES",
        texto,
        re.S
    )

    datos["programa_estudios"] = (
        programa.group(1).strip()
        if programa else None
    )

    return datos

In [6]:
def extraer_periodos(texto):

    patron = (
        r"Periodo Académico\s+Ciclo\s+"
        r"(\d{4}-[IV]+)\s+([IVX]+)"
    )

    return re.findall(
        patron,
        texto
    )

In [7]:
def evaluar_trayectoria(periodos):

    if len(periodos) >= 2:
        return "Incluido"

    return "Excluir"

In [8]:
def extraer_cursos(bloque, periodo, ciclo):

    cursos = []

    bloque = bloque.replace("\n", " ")
    bloque = re.sub(
        r"\s+",
        " ",
        bloque
    )

    patron = (
        r"\d+\s+(.*?)\s+"
        r"(PREVIO AL INICIO|LOGRADO|EN PROCESO|INICIO)"
        r"\s+(\d+)\s+(\d+)\s+(\d+)"
    )


    resultados = re.findall(
        patron,
        bloque
    )


    for curso, estado, nota, credito, puntaje in resultados:

        cursos.append({

            "periodo": periodo,
            "ciclo": ciclo,
            "curso": curso.strip(),
            "estado": estado,
            "nota": int(nota),
            "credito": int(credito),
            "puntaje": int(puntaje)

        })


    return cursos

In [9]:
def extraer_registros_academicos(texto):

    registros = []

    patron = (
        r"Periodo Académico\s+Ciclo\s+"
        r"(\d{4}-[IV]+)\s+([IVX]+)"
        r"(.*?)(?=Periodo Académico|$)"
    )


    periodos = re.findall(
        patron,
        texto,
        re.S
    )


    for periodo, ciclo, contenido in periodos:

        cursos = extraer_cursos(
            contenido,
            periodo,
            ciclo
        )

        registros.extend(cursos)


    return registros

In [10]:
def procesar_pdf(ruta_pdf):

    texto = leer_pdf(ruta_pdf)

    datos = extraer_datos_generales(texto)

    periodos = extraer_periodos(texto)


    if evaluar_trayectoria(periodos) == "Excluir":
        return [], {
            **datos,
            "cantidad_periodos": len(periodos),
            "decision": "Excluir"
        }


    registros = extraer_registros_academicos(texto)


    for registro in registros:
        registro.update(datos)


    resumen = {
        **datos,
        "cantidad_periodos": len(periodos),
        "decision": "Incluido"
    }


    return registros, resumen

In [11]:
todos_los_registros = []
resumen_estudiantes = []


for i, archivo in enumerate(archivos):

    print(
        f"Procesando {i+1}/{len(archivos)}: {archivo.name}"
    )


    registros, resumen = procesar_pdf(
        archivo
    )


    todos_los_registros.extend(
        registros
    )

    resumen_estudiantes.append(
        resumen
    )

Procesando 1/247: 04041859.pdf
Procesando 2/247: 10610527.pdf
Procesando 3/247: 20107537.pdf
Procesando 4/247: 20893955.pdf
Procesando 5/247: 21063279.pdf
Procesando 6/247: 21076480.pdf
Procesando 7/247: 21087602.pdf
Procesando 8/247: 21121175.pdf
Procesando 9/247: 21123566.pdf
Procesando 10/247: 21124788.pdf
Procesando 11/247: 21124939.pdf
Procesando 12/247: 21125146.pdf
Procesando 13/247: 21128581.pdf
Procesando 14/247: 21134535.pdf
Procesando 15/247: 21135881.pdf
Procesando 16/247: 21137050.pdf
Procesando 17/247: 21137674.pdf
Procesando 18/247: 21140269.pdf
Procesando 19/247: 40143017.pdf
Procesando 20/247: 40327272.pdf
Procesando 21/247: 40386407.pdf
Procesando 22/247: 40681363.pdf
Procesando 23/247: 40803998.pdf
Procesando 24/247: 41190593.pdf
Procesando 25/247: 42157445.pdf
Procesando 26/247: 42161435.pdf
Procesando 27/247: 42201929.pdf
Procesando 28/247: 42278910.pdf
Procesando 29/247: 42595681.pdf
Procesando 30/247: 42848467.pdf
Procesando 31/247: 42880150.pdf
Procesando 32/247

In [12]:
df_registros = pd.DataFrame(
    todos_los_registros
)

df_registros.head()

,periodo,ciclo,curso,estado,nota,credito,puntaje,id_estudiante,programa_estudios
0,2024-I,I,DESARROLLO PERSONAL I,LOGRADO,15,3,45,04041859,EDUCACIÓN PRIMARIA (R.V.M N° 204 - 2019 - MINEDU)
1,2024-I,I,DESARROLLO Y APRENDIZAJE I,EN PROCESO,13,5,65,04041859,EDUCACIÓN PRIMARIA (R.V.M N° 204 - 2019 - MINEDU)
2,2024-I,I,FUNDAMENTOS PARA LA EDUCACIÓN PRIMARIA,LOGRADO,15,5,75,04041859,EDUCACIÓN PRIMARIA (R.V.M N° 204 - 2019 - MINEDU)
3,2024-I,I,LECTURA Y ESCRITURA EN LA EDUCACIÓN SUPERIOR,LOGRADO,15,3,45,04041859,EDUCACIÓN PRIMARIA (R.V.M N° 204 - 2019 - MINEDU)
4,2024-I,I,PRÁCTICA E INVESTIGACIÓN I,EN PROCESO,13,5,65,04041859,EDUCACIÓN PRIMARIA (R.V.M N° 204 - 2019 - MINEDU)


In [13]:
df_resumen = pd.DataFrame(
    resumen_estudiantes
)

df_resumen.head()

,id_estudiante,programa_estudios,cantidad_periodos,decision
0,04041859,EDUCACIÓN PRIMARIA (R.V.M N° 204 - 2019 - MINEDU),5,Incluido
1,10610527,COMPUTACIÓN E INFORMÁTICA (R.D. N° 0223 - 2012...,5,Incluido
2,20107537,EDUCACIÓN PRIMARIA (R.V.M N° 204 - 2019 - MINEDU),5,Incluido
3,20893955,COMUNICACIÓN (R.V.M N° 143 - 2020 - MINEDU),2,Incluido
4,21063279,EDUCACIÓN FÍSICA (R.V.M N° 147 - 2020 - MINEDU),8,Incluido


In [14]:
df_registros.to_csv(
    DATA_INTERIM / "registros_academicos_raw.csv",
    index=False,
    encoding="utf-8-sig"
)


df_resumen.to_csv(
    DATA_INTERIM / "resumen_estudiantes.csv",
    index=False,
    encoding="utf-8-sig"
)